# KADMON — Export QuickBundles + Partial OT aligné

Ce notebook reprend le pipeline du notebook `00` et exporte deux tableaux de même forme. Pour chaque indice `i`, `source_matched[i]` correspond à la projection barycentrique `target_matched[i]` déterminée par le plan OT.

In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE if (HERE / "kadmon").is_dir() else HERE.parent
sys.path.insert(0, str(PROJECT_ROOT))
BUNDLES_DIR = PROJECT_ROOT / "notebooks" / "bundles"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "centroids_export"

MODE = "all"  # "pair" ou "all"
SOURCE_SUBJECT = "103818"
TARGET_SUBJECT = "135528"
INPUT_GLOB = "*_12mpts_rasmm.npy"
SOURCE_PATH = BUNDLES_DIR / SOURCE_SUBJECT / "nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m_12mpts_rasmm.npy"
TARGET_PATH = BUNDLES_DIR / TARGET_SUBJECT / "nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m_12mpts_rasmm.npy"

N_POINTS = 12
QUICKBUNDLES_THRESHOLD_MM = 7.0
PARTIAL_OT_MASS = 0.99
MAX_REPRESENTATIVES = 5000
MAX_COST_MATRIX_BYTES = 512 * 2**20

In [2]:
def bundle_key(path):
    suffix = f"_{N_POINTS}mpts_rasmm.npy"
    return path.name[:-len(suffix)] if path.name.endswith(suffix) else path.stem

if MODE == "pair":
    jobs = [(bundle_key(SOURCE_PATH), SOURCE_PATH, TARGET_PATH)]
elif MODE == "all":
    source_paths = (BUNDLES_DIR / SOURCE_SUBJECT / "nn_8mm").glob(INPUT_GLOB)
    target_paths = (BUNDLES_DIR / TARGET_SUBJECT / "nn_8mm").glob(INPUT_GLOB)
    source_index = {bundle_key(p): p for p in source_paths}
    target_index = {bundle_key(p): p for p in target_paths}
    names = sorted(source_index.keys() & target_index.keys())
    jobs = [(name, source_index[name], target_index[name]) for name in names]
else:
    raise ValueError("MODE doit être 'pair' ou 'all'.")

if not jobs:
    raise FileNotFoundError("Aucun bundle commun trouvé.")
print(f"{len(jobs)} paire(s) : {SOURCE_SUBJECT} → {TARGET_SUBJECT}")

31 paire(s) : 103818 → 135528


In [3]:
import numpy as np
from kadmon.comparison import compare_bundles
from kadmon.io import load_bundle

export_results = []
output_root = OUTPUT_DIR / f"{SOURCE_SUBJECT}_to_{TARGET_SUBJECT}"
for name, source_path, target_path in jobs:
    source = load_bundle(source_path, n_points=N_POINTS)
    target = load_bundle(target_path, n_points=N_POINTS)
    result = compare_bundles(
        source, target, compression="quickbundles", transport="partial",
        compression_parameters={"threshold": QUICKBUNDLES_THRESHOLD_MM},
        transport_parameters={"mass": PARTIAL_OT_MASS},
        max_representatives=MAX_REPRESENTATIVES,
        max_cost_matrix_bytes=MAX_COST_MATRIX_BYTES,
    )
    valid = np.asarray(result["barycentric"]["valid_mask"], dtype=bool)
    source_matched = np.asarray(result["source_representatives"])[valid]
    target_matched = np.asarray(result["barycentric"]["projection"])[valid]
    assert source_matched.shape == target_matched.shape
    assert np.isfinite(source_matched).all() and np.isfinite(target_matched).all()

    destination = output_root / name
    destination.mkdir(parents=True, exist_ok=True)
    np.save(destination / "source_matched.npy", source_matched)
    np.save(destination / "target_matched.npy", target_matched)
    export_results.append((name, source_matched.shape))
    print(f"{name}: {source_matched.shape[0]} paires, déplacement moyen={result['metrics']['mean_mm']:.3f} mm")
    del source, target, result, source_matched, target_matched

print(f"Export terminé : {output_root}")

Info: some functions in tractosearch.resampling are faster when 'numba' is installed
tractosearch_nn_8_0mm_all_AF_L_m: 532 paires, déplacement moyen=6.644 mm
tractosearch_nn_8_0mm_all_AF_R_m: 677 paires, déplacement moyen=10.331 mm
tractosearch_nn_8_0mm_all_CC_1_m: 90 paires, déplacement moyen=5.327 mm
tractosearch_nn_8_0mm_all_CC_2a_m: 343 paires, déplacement moyen=5.991 mm
tractosearch_nn_8_0mm_all_CC_2b_m: 329 paires, déplacement moyen=4.112 mm
tractosearch_nn_8_0mm_all_CC_3_m: 319 paires, déplacement moyen=5.319 mm
tractosearch_nn_8_0mm_all_CC_4_m: 226 paires, déplacement moyen=6.724 mm
tractosearch_nn_8_0mm_all_CC_5_m: 369 paires, déplacement moyen=6.880 mm
tractosearch_nn_8_0mm_all_CC_6_m: 781 paires, déplacement moyen=9.190 mm
tractosearch_nn_8_0mm_all_CC_7_m: 1066 paires, déplacement moyen=8.811 mm
tractosearch_nn_8_0mm_all_CG_L_m: 308 paires, déplacement moyen=7.075 mm
tractosearch_nn_8_0mm_all_CG_R_m: 202 paires, déplacement moyen=10.445 mm
tractosearch_nn_8_0mm_all_CST_L_m: 

In [4]:
for name, expected_shape in export_results:
    folder = output_root / name
    source = np.load(folder / "source_matched.npy", mmap_mode="r")
    target = np.load(folder / "target_matched.npy", mmap_mode="r")
    assert source.shape == target.shape == expected_shape
print(f"Alignement vérifié pour {len(export_results)} bundle(s).")

Alignement vérifié pour 31 bundle(s).
